In [27]:
import os, sys, shutil
import pandas as pd
from tqdm.notebook import tqdm

In [2]:
sys.path.append("../training_data")

In [3]:
from utils.new_pdbs import Cif, Pdb

# ATLAS data

Assuming that the chain in MDAtlas is the `auth_asym_id`.

In [4]:
with open("2023_03_09_ATLAS_pdb.txt", "r") as f:
    mdatlas = f.read().split("\n")[:-1]

mdatlas, mdatlas[-1]

(['1r6w_A',
  '2y44_A',
  '1ux6_A',
  '1jnr_C',
  '3mx7_A',
  '1u55_A',
  '3k59_A',
  '3owt_A',
  '4zds_A',
  '3st1_A',
  '3mmh_B',
  '4obi_A',
  '4ojl_C',
  '2ie6_A',
  '3frr_A',
  '5dje_A',
  '3rt4_A',
  '3lui_B',
  '4ftf_A',
  '3nqi_A',
  '1qau_A',
  '8b3w_A',
  '1xsz_A',
  '4g94_B',
  '7n0j_A',
  '4exo_A',
  '1h16_A',
  '3e4w_B',
  '2fb5_B',
  '1ng6_A',
  '3kz7_A',
  '2p3p_A',
  '7w81_B',
  '1tuk_A',
  '5noh_B',
  '2qmj_A',
  '6exa_B',
  '3kef_B',
  '6l34_A',
  '1cvr_A',
  '5wfy_A',
  '4k2m_A',
  '3iis_M',
  '1fs1_C',
  '1c1k_A',
  '6xrx_A',
  '1wlg_A',
  '4qtq_A',
  '1l5o_A',
  '5j2l_B',
  '6in7_A',
  '3e2d_A',
  '3a3d_B',
  '1huf_A',
  '1wf3_A',
  '2hq4_B',
  '3dso_A',
  '1n7z_B',
  '1ugp_B',
  '4cjn_A',
  '2bmo_B',
  '2h2z_A',
  '1eb0_A',
  '7r79_B',
  '1r5l_A',
  '5njb_B',
  '5m45_K',
  '2o7o_A',
  '1v9m_A',
  '1l2w_J',
  '2vri_A',
  '2q9r_A',
  '2bf6_A',
  '7mi2_A',
  '2jhq_A',
  '1lr0_A',
  '2a6s_B',
  '1zw0_G',
  '4tsh_A',
  '7nru_A',
  '2qsk_A',
  '5wvo_C',
  '3a6d_F',
  '1

# Extra test set

In [5]:
features = pd.read_pickle("../training_data/7.Extra_set/features.pkl")
print(len(features))
features

9


{'7gqu':     Residues                                                          \
          pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
 0       7gqu               1             A           12            A   
 1       7gqu               1             A           13            A   
 2       7gqu               1             A           14            A   
 3       7gqu               1             A           15            A   
 4       7gqu               1             A           16            A   
 ..       ...             ...           ...          ...          ...   
 414     7gqu               1             A          426            A   
 415     7gqu               1             A          427            A   
 416     7gqu               1             A          428            A   
 417     7gqu               1             A          429            A   
 418     7gqu               1             A          430            A   
 
                                   Label 

## How many PDBs are in the extra test set?

May not be the relevant chain.

In [6]:
mdatlas_nochain = set(p.split("_")[0] for p in mdatlas)

In [7]:
[i for i in features if i in mdatlas_nochain]

[]

## Process

In [8]:
Cif.path = "../training_data/7.Extra_set/cifs"
Cif.original_cifs_path = "../training_data/7.Extra_set/origcifs"

In [9]:
ets = {
    (pdb, cif.residues.label_asym_id.unique().item()): {
        u: ures
        for u, ures in cif.residues.groupby("pdbx_sifts_xref_db_acc", sort=False)
        if u != "?"
    }
    for pdb in features
    for cif in (Cif(pdb),)
}

ets

{('7gqu',
  'A'): {'Q14191':      label_comp_id label_asym_id label_entity_id label_seq_id  \
  0              PHE             A               1           12   
  11             LEU             A               1           13   
  19             TRP             A               1           14   
  33             PRO             A               1           15   
  40             ALA             A               1           16   
  ...            ...           ...             ...          ...   
  3299           SER             A               1          426   
  3305           ARG             A               1          427   
  3316           LEU             A               1          428   
  3324           ASP             A               1          429   
  3332           HIS             A               1          430   
  
       pdbx_PDB_ins_code auth_seq_id auth_comp_id auth_asym_id  \
  0                    ?         527          PHE            A   
  11                   ?         5

In [10]:
{k: (len(v), tuple(len(r) for r in v.values())) for k, v in ets.items()}

{('7gqu', 'A'): (1, (419,)),
 ('7yg5', 'A'): (1, (1319,)),
 ('8aq6', 'G'): (1, (169,)),
 ('8f4s', 'A'): (1, (298,)),
 ('8jp0', 'A'): (1, (719,)),
 ('8qni', 'A'): (1, (377,)),
 ('8uk6', 'A'): (1, (573,)),
 ('8v81', 'A'): (1, (1132,)),
 ('9dnm', 'C'): (2, (329, 106))}

### 9dnm

**9dnm is a chimera with a cytochrome for crystallization**

In [11]:
ets[("9dnm", "C")].keys()

dict_keys(['P29066', 'P0ABE7'])

In [12]:
ets[("9dnm", "C")].pop('P0ABE7')
ets[("9dnm", "C")].keys()

dict_keys(['P29066'])

<br>

In [13]:
tuple((k, u) for k, v in ets.items() for u in v)

((('7gqu', 'A'), 'Q14191'),
 (('7yg5', 'A'), 'Q15878'),
 (('8aq6', 'G'), 'Q9GV45'),
 (('8f4s', 'A'), 'P0DTD1'),
 (('8jp0', 'A'), 'P32418-2'),
 (('8qni', 'A'), 'Q13191'),
 (('8uk6', 'A'), 'A0A1D8PQM9'),
 (('8v81', 'A'), 'P13569'),
 (('9dnm', 'C'), 'P29066'))

### 8jp0

**In the RCSB PDB 8jp0 is associated to P32418**, which was also force to successfully find apo structures.

In [14]:
ets[('8jp0', 'A')] = {'P32418': ets.pop(('8jp0', 'A'))['P32418-2']}

In [15]:
ets[('8jp0', 'A')]['P32418'].replace(to_replace={"pdbx_sifts_xref_db_acc": "P32418-2"}, value="P32418", inplace=True)
ets[('8jp0', 'A')]['P32418']

,label_comp_id,label_asym_id,label_entity_id,label_seq_id,pdbx_PDB_ins_code,auth_seq_id,auth_comp_id,auth_asym_id,pdbx_PDB_model_num,pdbx_label_index,pdbx_sifts_xref_db_name,pdbx_sifts_xref_db_acc,pdbx_sifts_xref_db_num,pdbx_sifts_xref_db_res
0,GLY,A,1,51,?,51,GLY,A,1,51,UNP,P32418,51,G
4,SER,A,1,52,?,52,SER,A,1,52,UNP,P32418,52,S
10,TYR,A,1,53,?,53,TYR,A,1,53,UNP,P32418,53,Y
22,TYR,A,1,54,?,54,TYR,A,1,54,UNP,P32418,54,Y
34,CYS,A,1,55,?,55,CYS,A,1,55,UNP,P32418,55,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5613,HIS,A,1,933,?,933,HIS,A,1,933,UNP,P32418,933,H
5623,ILE,A,1,934,?,934,ILE,A,1,934,UNP,P32418,934,I
5631,LYS,A,1,935,?,935,LYS,A,1,935,UNP,P32418,935,K
5640,GLY,A,1,936,?,936,GLY,A,1,936,UNP,P32418,936,G


In [16]:
tuple((k, u) for k, v in ets.items() for u in v)

((('7gqu', 'A'), 'Q14191'),
 (('7yg5', 'A'), 'Q15878'),
 (('8aq6', 'G'), 'Q9GV45'),
 (('8f4s', 'A'), 'P0DTD1'),
 (('8qni', 'A'), 'Q13191'),
 (('8uk6', 'A'), 'A0A1D8PQM9'),
 (('8v81', 'A'), 'P13569'),
 (('9dnm', 'C'), 'P29066'),
 (('8jp0', 'A'), 'P32418'))

In [17]:
ets_ups = {
    u: {
        "pdb": pdb, 
        "label_asym_id": chain,
        "ures": ures
    }
    for (pdb, chain), ud in ets.items()
    for u, ures in ud.items()    
}

ets_ups

{'Q14191': {'pdb': '7gqu',
  'label_asym_id': 'A',
  'ures':      label_comp_id label_asym_id label_entity_id label_seq_id  \
  0              PHE             A               1           12   
  11             LEU             A               1           13   
  19             TRP             A               1           14   
  33             PRO             A               1           15   
  40             ALA             A               1           16   
  ...            ...           ...             ...          ...   
  3299           SER             A               1          426   
  3305           ARG             A               1          427   
  3316           LEU             A               1          428   
  3324           ASP             A               1          429   
  3332           HIS             A               1          430   
  
       pdbx_PDB_ins_code auth_seq_id auth_comp_id auth_asym_id  \
  0                    ?         527          PHE            A   
  

# ATLAS

## Download cifs

In [22]:
# Recycling check
[f[:4] for f in os.listdir("mdatlascifs") if f.endswith(".cif.gz") and f[:4] not in mdatlas_nochain]

[]

In [23]:
for pdb in mdatlas_nochain:
    fn = f"mdatlascifs/{pdb}_updated.cif.gz"
    if not os.path.isfile(fn):
        if os.path.isfile(fn.replace("_updated", "")):
            shutil.move(fn.replace("_updated", ""), fn)
        else:
            with open(fn, "wb") as f:
                f.write(Pdb(pdb).cif._cif_content)

## Process

In [24]:
Cif.path = "mdatlascifs"
Cif.original_cifs_path = "mdatlascifs"

In [28]:
matches = []

for i in tqdm(mdatlas):
    pdb, chain = i.split("_")
    cif = Cif(pdb, filename=f"mdatlascifs/{pdb}_updated.cif.gz")
    chainres = cif.residues.query(f"auth_asym_id == '{chain}'")
    
    if "pdbx_sifts_xref_db_acc" not in chainres.columns or not any(u in ets_ups for u in chainres.pdbx_sifts_xref_db_acc.unique()):
        continue

    for u, ures in chainres.groupby("pdbx_sifts_xref_db_acc", sort=False):
        if u != "?":
            orig = ets_ups.get(u, {"pdb": "", "label_asym_id": "", "ures": pd.DataFrame(columns=["pdbx_sifts_xref_db_acc", "pdbx_sifts_xref_db_num"])})
            merge = len(
                ures
                .merge(
                    orig["ures"],
                    on = ["pdbx_sifts_xref_db_acc", "pdbx_sifts_xref_db_num"],
                    how = "outer", indicator = True
                ).query("_merge == 'both'")
            )
            matches.append({
                "orig_pdb": orig["pdb"],
                "orig_label_asym_id": orig["label_asym_id"],
                "orig_u": u,
                "orig_umin": orig["ures"]["pdbx_sifts_xref_db_num"].astype(int).min(),
                "orig_umax": orig["ures"]["pdbx_sifts_xref_db_num"].astype(int).max(),
                "pdb": pdb,
                "chain": chain,
                "label_asym_id": ures.label_asym_id.unique().item(),
                "umin": ures["pdbx_sifts_xref_db_num"].astype(int).min(),
                "umax": ures["pdbx_sifts_xref_db_num"].astype(int).max(),
                "overlap_ets": merge/( len(orig["ures"]) ),
                "overlap_atlas": merge/( len(ures) ),
            })

  0%|          | 0/1938 [00:00<?, ?it/s]

In [29]:
matchesdf = pd.DataFrame(matches)
matchesdf.to_pickle("atlas_matches.pkl")
matchesdf

,orig_pdb,orig_label_asym_id,orig_u,orig_umin,orig_umax,pdb,chain,label_asym_id,umin,umax,overlap_ets,overlap_atlas
0,8f4s,A,P0DTD1,6799,7096,6wqd,B,B,4020,4140,0.0,0.0
1,8f4s,A,P0DTD1,6799,7096,6zpe,A,A,4263,4383,0.0,0.0
2,8f4s,A,P0DTD1,6799,7096,7k7p,B,A,10,126,0.0,0.0
3,8f4s,A,P0DTD1,6799,7096,7buy,A,A,3264,3569,0.0,0.0
4,8f4s,A,P0DTD1,6799,7096,6zsl,B,A,5326,5917,0.0,0.0
5,8f4s,A,P0DTD1,6799,7096,6yhu,B,B,4018,4134,0.0,0.0
6,8f4s,A,P0DTD1,6799,7096,5rlk,B,B,5326,5917,0.0,0.0
